# Visualization of the Trajectories predicted by the Model

In [1]:
import os

import torch

import numpy as np
import pandas as pd

import plotly.express as px

from neural_decode.dataset.transformer_dataloader import get_train_val_loaders

from neural_decode.models.hopfield_only import HopfieldOnlyDecoder
from neural_decode.models.transformer import TransformerNeuralDecoder 
from neural_decode.models.transformer_hopfield import TransformerHopfieldDecoder

from neural_decode.post_hoc_analysis.embedding_analysis import compute_umap_embeddings
from neural_decode.post_hoc_analysis.saliency_based_analysis import compute_and_return_attribution_maps

/Users/johannesbauer/opt/anaconda3/envs/streamlit_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def convert_velocity_to_path(velocity_lst, eps = 0.05, lb = "model"):
    dictionary_of_trajectories = {"time": [0],
                                  "x_pos": [0],
                                  "y_pos": [0],
                                  "x_vel": [0],
                                  "y_vel": [0],
                                  "label": []}
    
    x_pos = 0
    y_pos = 0



    for t in range(len(velocity_lst)):

        x_vel = velocity_lst[t][0]
        y_vel = velocity_lst[t][1]

        x_pos = x_pos + (eps*x_vel)
        y_pos = y_pos + (eps*y_vel)


        dictionary_of_trajectories["time"].append(t + 1)
        dictionary_of_trajectories["x_pos"].append(x_pos)
        dictionary_of_trajectories["y_pos"].append(y_pos)
        dictionary_of_trajectories["x_vel"].append(x_vel)
        dictionary_of_trajectories["y_vel"].append(y_vel)

    dictionary_of_trajectories["label"] += [lb]*len(dictionary_of_trajectories["time"])

    return dictionary_of_trajectories


In [3]:
base_dir = os.getcwd().split("additional_scripts")[0]
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using: {device}")

Using: cpu


In [4]:
path_to_neural_dataset = os.path.join(base_dir, "data", "perich_miller_population_2018", "t_20130819_center_out_reaching")
train_dataset, train_loader, val_dataset, val_loader = get_train_val_loaders(path_to_neural_dataset, batch_size=64)

num_units = len(train_dataset.get_unit_ids())
print(f"Num Units in Session: {num_units}")

Num Units in Session: 55


In [5]:
path_to_neural_dataset = os.path.join(base_dir, "data", "perich_miller_population_2018", "t_20130819_center_out_reaching")
train_dataset, train_loader, val_dataset, val_loader = get_train_val_loaders(path_to_neural_dataset, batch_size=64)

num_units = len(train_dataset.get_unit_ids())
print(f"Num Units in Session: {num_units}")

Num Units in Session: 55


In [6]:
path_to_models = os.path.join(base_dir, "results", "models")

tf_save_file = os.path.join(path_to_models, "transformer_model.pth")
hopfield_only_save_file = os.path.join(path_to_models, "hopfield_only_model.pth")
tf_hopfield_save_file = os.path.join(path_to_models, "transformer_hopfield_model.pth")

In [7]:
def move_to_gpu(data, device):
    """
    Recursively moves tensors (or collections of tensors) to the given device.
    """
    if isinstance(data, torch.Tensor):
        return data.to(device)
    elif isinstance(data, dict):
        return {k: move_to_gpu(v, device) for k, v in data.items()}
    elif isinstance(data, list):
        return [move_to_gpu(elem, device) for elem in data]
    else:
        return data

def compute_predictions(dataloader, model, device = "cpu"):
    '''
    Computes the R^2 scores given a dataloader, model, and device. Note that this is the main function which should be use for evlauating the models, not the r2_score function directly.
    '''
    # Compute R2 score over the entire dataset
    total_target = []
    total_pred = []
    for batch in dataloader:
        batch = move_to_gpu(batch, device)
        pred = model(**batch["model_inputs"])
        target = batch["target_values"]

        total_target.append(target)
        total_pred.append(pred)

    # Concatenate all batch outputs
    total_target = torch.cat(total_target).detach().numpy()
    total_pred = torch.cat(total_pred).detach().numpy()


    return total_pred, total_target

In [8]:
tf_model = TransformerNeuralDecoder(
    num_units=num_units,
    bin_size=10e-3,
    sequence_length=1.0,
    dim_output=2,
    dim_hidden=128,
    n_layers=3,
    n_heads=4,
).to(device)



# Load state dicts for both models.
tf_model.load_state_dict(torch.load(tf_save_file, map_location=device))

# Add tokenizer for train and validation loader.
train_dataset.transform = tf_model.tokenize
val_dataset.transform = tf_model.tokenize

tf_target, predicted = compute_predictions(val_loader, tf_model, "cpu")


Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [9]:
hopfield_only_model = HopfieldOnlyDecoder(
    num_units=num_units,
    bin_size=10e-3,
    sequence_length=1.0,
    dim_output=2,
    dim_hidden=128,
    n_layers=3,
).to(device)

# Load state dicts for both models.
hopfield_only_model.load_state_dict(torch.load(hopfield_only_save_file, map_location=device))

# Add tokenizer for train and validation loader.
train_dataset.transform = hopfield_only_model.tokenize
val_dataset.transform = hopfield_only_model.tokenize

hopfield_only_target, predicted = compute_predictions(val_loader, hopfield_only_model, "cpu")

In [10]:
tf_hopfield_model = TransformerHopfieldDecoder(
    num_units=num_units,
    bin_size=10e-3,
    sequence_length=1.0,
    dim_output=2,
    dim_hidden=128,
    n_layers=3,
    n_heads=4,
).to(device)

tf_hopfield_model.load_state_dict(torch.load(tf_hopfield_save_file, map_location=device))

train_dataset.transform = tf_hopfield_model.tokenize
val_dataset.transform = tf_hopfield_model.tokenize

tf_hopfield_target, predicted = compute_predictions(val_loader, tf_hopfield_model, "cpu")

In [51]:
out_tf = convert_velocity_to_path(tf_target[0], 0.01, "Transformer")
out_hopfield = convert_velocity_to_path(hopfield_only_target[0], 0.01, "Hopfield Only")
out_tf_hopfield = convert_velocity_to_path(tf_hopfield_target[0], 0.01, "Transformer + Hopfield Only")
out_pred = convert_velocity_to_path(predicted[0], 0.01, "Truth")

In [52]:
final_pd = pd.concat([pd.DataFrame(out_tf), 
                      pd.DataFrame(out_hopfield),
                      pd.DataFrame(out_tf_hopfield),
                    pd.DataFrame(out_pred)
                    ])

In [53]:
final_pd.to_csv("/Users/johannesbauer/Documents/Coding/neuro_comp_project/results/evaluations/bad_example_of_pos")

In [54]:
out_pd = pd.DataFrame(out_tf)

fig = px.scatter(final_pd, x = "x_pos", y = "y_pos", animation_frame="time", color="label")
fig.update_traces(marker=dict(size=15))  
fig.update_xaxes(range=[-5, 5], constrain='domain')
fig.update_yaxes(range=[-3, 3], scaleanchor='x', scaleratio=1)